# Финальная модель бота
По ноутбуку 14: сумму кредита чинит только сочетание «без абсолютных сумм + монотонные ограничения» (V3).
Добавляем ещё одно ограничение: риск не растёт с числом кредитов в истории (на dev при 0 – 1 кредите вклад
+0.29, при 9+ кредитах −0.29), это V4.

Здесь:
1. Сравнение V0, V3 и V4 по AUC на тех же фолдах.
2. Проверка «что если» только в пределах диапазонов, на которых училась модель (вне диапазона бот оценку
   давать не будет). Тест «платёж при той же сумме» переведён в информативный: рост платежа одновременно
   увеличивает нагрузку и сокращает срок, эффекты противоположны.
3. Обучение выбранного варианта, отчёт на hold-out, причины отказа, сценарии.
4. Сохранение модели и конфигурации для бота.

Выбор варианта делается по dev. Hold-out только отчёт.

In [ ]:
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

sys.path.insert(0, "..")
from src.bot_features import BOT_FEATURES, answers_to_features
from src.data import load_features, split_dev_holdout
from src.explain import MIN_REASON_SHAP, group_shap, top_reasons
from src.features import ID_COL, TARGET_COL
from src.metrics import calibration_report, calibration_table, ks_stat
from src.models import cv_predict, make_bot_model, to_lgbm

PROCESSED = Path("../data/processed")
MODELS = Path("../models")
pd.set_option("display.width", 170)

df = load_features(PROCESSED / "train_features.parquet")
dev, hold = split_dev_holdout(df, PROCESSED / "holdout_ids.csv")
y = dev[TARGET_COL]
y_h = hold[TARGET_COL].values
decision = json.loads((MODELS / "decision_config.json").read_text())
R, T = decision["cost_ratio"], decision["threshold"]
limits = json.loads((MODELS / "bot_input_ranges.json").read_text())

## 1. Варианты

In [ ]:
ABS_AMOUNTS = ["AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE"]
NO_ABS = [c for c in BOT_FEATURES if c not in ABS_AMOUNTS]
MONO_V3 = {
    "CREDIT_INCOME_RATIO": 1, "ANNUITY_INCOME_RATIO": 1,
    "BUREAU_HAS_OVERDUE_SUM": 1, "BUREAU_IS_ACTIVE_SUM": 1,
    "BUREAU_AMT_CREDIT_SUM_DEBT_SUM": 1, "BUREAU_LAST1Y_COUNT": 1,
    "DAYS_EMPLOYED": 1, "AMT_INCOME_TOTAL": -1,
}
MONO_V4 = {**MONO_V3, "BUREAU_COUNT": -1}

V0 = "V0: как сейчас"
V3 = "V3: без абс. сумм + монотонность"
V4 = "V4: V3 + монотонность по числу кредитов"
variants = {V0: (BOT_FEATURES, None), V3: (NO_ABS, MONO_V3), V4: (NO_ABS, MONO_V4)}

results = {}
for name, (cols, mono) in variants.items():
    oof, fa = cv_predict(lambda cols=cols, mono=mono: make_bot_model(cols, mono), to_lgbm(dev[cols]), y)
    results[name] = (oof, fa)
    print(f"{name}: AUC {fa.mean():.4f} ± {fa.std():.4f}")

In [ ]:
rows = []
for name, (oof, fa) in results.items():
    d0, d3 = fa - results[V0][1], fa - results[V3][1]
    cal = calibration_report(name, y.values, oof)
    rows.append({"вариант": name, "ROC AUC": round(fa.mean(), 4), "Gini": round(2 * fa.mean() - 1, 4),
                 "ECE": cal["ECE"],
                 "diff vs V0 (mean / min / max)": f"{d0.mean():+.4f} / {d0.min():+.4f} / {d0.max():+.4f}",
                 "diff vs V3 (mean / min / max)": f"{d3.mean():+.4f} / {d3.min():+.4f} / {d3.max():+.4f}"})
pd.DataFrame(rows).set_index("вариант")

## 2. «Что если» в пределах диапазонов обучения

In [ ]:
fitted = {name: make_bot_model(cols, mono).fit(to_lgbm(dev[cols]), y) for name, (cols, mono) in variants.items()}

base = {"income": 150_000, "credit": 400_000, "annuity": 20_000, "goods_price": 400_000,
        "income_type": "Working", "education": "Secondary / secondary special", "employment_years": 5,
        "own_car": "N", "car_age": None, "own_realty": "Y", "contract_type": "Cash loans",
        "bureau_count": 2, "bureau_active": 1, "bureau_debt": 60_000, "bureau_overdue": False, "bureau_last_year": 0}


def out_of_range(answers):
    f = answers_to_features(answers).iloc[0].to_dict()
    f["EMPLOYMENT_YEARS"] = -f["DAYS_EMPLOYED"] / 365 if pd.notna(f["DAYS_EMPLOYED"]) else np.nan
    return [c for c, lim in limits.items() if c in f and pd.notna(f[c]) and not (lim["min"] <= f[c] <= lim["max"])]


K = [0.5, 1.0, 1.5, 2.0, 3.0]
sweeps = {  # название: (изменения ответов, ожидаемое направление риска или "info")
    "сумма кредита x K (платёж и цена товара пропорционально)":
        ([dict(credit=400_000 * k, annuity=20_000 * k, goods_price=400_000 * k) for k in K], "up"),
    "доход x K": ([dict(income=150_000 * k) for k in K], "down"),
    "стаж, лет: 0.5 / 1 / 3 / 5 / 10 / 20": ([dict(employment_years=k) for k in [0.5, 1, 3, 5, 10, 20]], "down"),
    "активных кредитов 0..5 из 5": ([dict(bureau_count=5, bureau_active=k) for k in range(6)], "up"),
    "всего кредитов в истории 1 / 2 / 4 / 7 / 12 (активен 1)":
        ([dict(bureau_count=c, bureau_active=1) for c in [1, 2, 4, 7, 12]], "down"),
    "долг в других банках: 0 / 50 тыс. / 150 тыс. / 500 тыс. / 1.5 млн":
        ([dict(bureau_count=3, bureau_active=2, bureau_debt=v) for v in [0, 50_000, 150_000, 500_000, 1_500_000]], "up"),
    "просрочки: нет / да": ([dict(bureau_overdue=False), dict(bureau_overdue=True)], "up"),
    "(информативно) платёж x K при той же сумме": ([dict(annuity=20_000 * k) for k in K], "info"),
}


def probs(model, cols, changes):
    return [float(model.predict_proba(to_lgbm(answers_to_features({**base, **c}, features=cols)))[:, 1][0])
            for c in changes]


def verdict(ps, direction, tol=0.0005):
    if direction == "info":
        return "информативно"
    d = np.diff(ps) if direction == "up" else -np.diff(ps)
    return "ok" if (d >= -tol).all() else "НАРУШЕНИЕ"


table, detail, dropped = {}, {}, {}
for sname, (changes, direction) in sweeps.items():
    valid = [c for c in changes if not out_of_range({**base, **c})]
    dropped[sname] = len(changes) - len(valid)
    for vname, (cols, _) in variants.items():
        ps = probs(fitted[vname], cols, valid)
        detail[(sname, vname)] = ps
        table.setdefault(sname, {})[vname] = verdict(ps, direction)
out = pd.DataFrame(table).T
out["точек вне диапазона"] = pd.Series(dropped)
out

In [ ]:
for s in [list(sweeps)[0], list(sweeps)[-1], list(sweeps)[4]]:
    print(s)
    n = len(detail[(s, V0)])
    print(pd.DataFrame({v: [f"{p:.1%}" for p in detail[(s, v)]] for v in variants}, index=range(1, n + 1)))
    print()

## 3. Выбранный вариант: hold-out, причины, сценарии

In [ ]:
CHOSEN = V4
cols, mono = variants[CHOSEN]
model = fitted[CHOSEN]
oof = results[CHOSEN][0]
X_h = to_lgbm(hold[cols])
p_h = model.predict_proba(X_h)[:, 1]


def decide(y_, p, R=R):
    reject = p >= 1 / (1 + R)
    fn = int(((~reject) & (y_ == 1)).sum())
    fp = int((reject & (y_ == 0)).sum())
    loss = (R * fn + fp) / len(y_)
    naive = min(R * y_.mean(), 1 - y_.mean())
    return {"отказов, %": round(reject.mean() * 100, 1),
            "отсеяно дефолтов, %": round(((reject) & (y_ == 1)).sum() / (y_ == 1).sum() * 100, 1),
            "дефолт среди одобренных, %": round(fn / max((~reject).sum(), 1) * 100, 2),
            "экономия, %": round((1 - loss / naive) * 100, 1)}


rows = []
for name, yy, p in [("dev (OOF)", y.values, oof), ("hold-out", y_h, p_h)]:
    auc = roc_auc_score(yy, p)
    rows.append({"выборка": name, "ROC AUC": round(auc, 4), "Gini": round(2 * auc - 1, 4),
                 "PR AUC": round(average_precision_score(yy, p), 4), "KS": round(ks_stat(yy, p), 4),
                 **{k: v for k, v in calibration_report(name, yy, p).items() if k in ("mean pred", "ECE")},
                 **decide(yy, p)})
pd.DataFrame(rows).set_index("выборка")

In [ ]:
rng = np.random.default_rng(42)
n = len(y_h)
boot = [roc_auc_score(y_h[idx], p_h[idx]) for idx in (rng.integers(0, n, n) for _ in range(500))]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"AUC на hold-out {roc_auc_score(y_h, p_h):.4f}, 95% ДИ [{lo:.4f}; {hi:.4f}]")
calibration_table(y_h, p_h, bins=10).round(4)

In [ ]:
# Причины отказа на hold-out (новые группы: стаж и тип занятости разделены, кредитная история переименована)
sv = model.booster_.predict(X_h, pred_contrib=True)[:, :-1]
reasons = top_reasons(group_shap(sv, X_h.columns), k=3, min_shap=MIN_REASON_SHAP)
refused = np.where(p_h >= T)[0]
print(f"Отказано: {len(refused)} ({len(refused) / len(p_h):.1%})")
print(pd.Series([len(reasons[i]) for i in refused]).value_counts().sort_index().rename("клиентов"))
counts = pd.Series([t for i in refused for t, _ in reasons[i]]).value_counts()
print("\nЧастота причин среди отказанных, %:")
print((counts / len(refused) * 100).round(1))
for k in range(3):
    vals = np.array([reasons[i][k][1] for i in refused if len(reasons[i]) > k])
    print(f"Причина №{k + 1}: квантили вклада 10% / 50% / 90%: {np.quantile(vals, [0.1, 0.5, 0.9]).round(2)}")

In [ ]:
def assess(answers):
    X = to_lgbm(answers_to_features(answers, features=cols))
    p = float(model.predict_proba(X)[:, 1][0])
    contrib = model.booster_.predict(X, pred_contrib=True)[:, :-1]
    r = top_reasons(group_shap(contrib, X.columns), k=3, min_shap=MIN_REASON_SHAP)[0]
    return p, p >= T, r


profiles = {
    "типичный клиент": base,
    "высокая нагрузка, просрочки, много новых кредитов": {**base, "income": 90_000, "credit": 900_000, "annuity": 45_000,
        "goods_price": 800_000, "bureau_count": 7, "bureau_active": 4, "bureau_debt": 500_000,
        "bureau_overdue": True, "bureau_last_year": 3},
    "стабильный, без кредитной истории": {**base, "income": 300_000, "credit": 200_000, "annuity": 8_000,
        "goods_price": 200_000, "education": "Higher education", "employment_years": 12,
        "own_car": "Y", "car_age": 3, "bureau_count": 0},
    "пенсионер": {**base, "income_type": "Pensioner", "employment_years": None, "income": 120_000,
        "credit": 250_000, "annuity": 12_000, "goods_price": 250_000},
    "короткая история, мало опыта": {**base, "employment_years": 0.7, "bureau_count": 1, "bureau_active": 1,
        "bureau_debt": 30_000, "bureau_last_year": 1},
}
for name, ans in profiles.items():
    p, refuse, rs = assess(ans)
    print(f"--- {name}: вероятность дефолта {p:.1%} -> {'ОТКАЗ' if refuse else 'одобрить'}; вне диапазона: {out_of_range(ans) or 'нет'} ---")
    if refuse:
        for text, val in rs:
            print(f"  {text}  (вклад {val:+.2f})")
    print()

## 4. Сохраняем

In [ ]:
joblib.dump(model, MODELS / "lgbm_bot.joblib")
(MODELS / "features_bot.json").write_text(json.dumps(cols, ensure_ascii=False, indent=1))
bot_config = {
    "variant": CHOSEN, "cost_ratio": R, "threshold": T, "min_reason_shap": MIN_REASON_SHAP, "calibrated": False,
    "model": "models/lgbm_bot.joblib", "features": "models/features_bot.json",
    "input_ranges": "models/bot_input_ranges.json", "monotone_constraints": mono,
    "cv_auc_dev": round(float(results[CHOSEN][1].mean()), 4),
    "auc_holdout": round(float(roc_auc_score(y_h, p_h)), 4),
}
(MODELS / "decision_config_bot.json").write_text(json.dumps(bot_config, ensure_ascii=False, indent=2))
print("Сохранено: lgbm_bot.joblib, features_bot.json, decision_config_bot.json")

Что смотрим:
1. Цена V3 и V4 в AUC против V0 (min по фолдам) и между собой: даёт ли ограничение по числу кредитов что-то
   или стоит что-то.
2. Проходят ли V3 и V4 все сценарии без «НАРУШЕНИЕ» в пределах диапазонов; сколько точек вне диапазона.
3. Реакция на сумму: монотонна ли и не слишком ли слабая (V3 давал только 7.6% -> 8.4% от x0.5 до x3).
4. Hold-out (AUC, калибровка), число причин и их разнообразие после переименования и разделения групп.
5. Что видит бот у «короткой истории»: попадает ли причина «Небольшое число кредитов» и «Небольшой стаж».